# Analyse GIFT-Eval Benchmarks

This notebook compares local TimeCopilot GIFT-Eval runs against selected benchmark result files from a cloned `gift-eval` repository.

In [ ]:
from pathlib import Path

import pandas as pd

def resolve_existing_path(*candidates: Path) -> Path:
    for candidate in candidates:
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f"None of these paths exist: {[str(c) for c in candidates]}")


LOCAL_RESULTS_ROOT = resolve_existing_path(
    Path("experiments/gift-eval/results/timecopilot"),
    Path("../../experiments/gift-eval/results/timecopilot"),
)
BENCHMARK_RESULTS_ROOT = Path(
    "/Users/fearghalodonncha/Work/DigitalTwin/TSFM_Agent/Development/Benchmarks/gift-eval/results"
)

BENCHMARK_MODELS = [
    "PatchTST-FM-r1",
    "Granite-PatchTST-FM-r1",
    "FlowState-r1.1",
    "TTM-R2-Pretrained",
]

# Compare only the multi-dataset runs by default.
LOCAL_RUN_NAMES = [
    "flowstate-all",
    "ibm-r3-all",
    "ttm-r3-all",
]

# If you want to inspect the single-dataset TTM result separately, add:
# LOCAL_RUN_NAMES.append("ttm-r3")

DATASETS = None
# Example:
# DATASETS = ["m4_weekly/W/short", "m4_monthly/M/short"]

METRICS = [
    "eval_metrics/MASE[0.5]",
    "eval_metrics/mean_weighted_sum_quantile_loss",
    "eval_metrics/MAE[0.5]",
    "eval_metrics/RMSE[mean]",
]


In [ ]:
def read_results(csv_path: Path, run_name: str) -> pd.DataFrame:
    df = pd.read_csv(csv_path).copy()
    df["run_name"] = run_name
    return df.drop_duplicates(subset=["dataset", "model"], keep="last")


def normalize_local_run_name(rel_parent: Path) -> str:
    if str(rel_parent) == ".":
        return "root"
    parts = rel_parent.parts
    if parts and parts[0].endswith("-all"):
        return parts[0]
    return rel_parent.as_posix()


def discover_local_results(results_root: Path, run_names: list[str] | None = None) -> pd.DataFrame:
    csv_paths = sorted(results_root.glob("**/all_results.csv"))
    if not csv_paths:
        raise ValueError(f"No local all_results.csv files found under {results_root}")
    allowed_runs = set(run_names) if run_names is not None else None
    dfs = []
    for csv_path in csv_paths:
        rel_parent = csv_path.parent.relative_to(results_root)
        run_name = normalize_local_run_name(rel_parent)
        if allowed_runs is not None and run_name not in allowed_runs:
            continue
        run_name = f"TC_{run_name}"
        dfs.append(read_results(csv_path, run_name=run_name))
    return pd.concat(dfs, ignore_index=True)


def load_benchmark_results(results_root: Path, model_names: list[str]) -> pd.DataFrame:
    dfs = []
    for model_name in model_names:
        csv_path = results_root / model_name / "all_results.csv"
        if not csv_path.exists():
            raise FileNotFoundError(f"Could not find benchmark results for {model_name}: {csv_path}")
        dfs.append(read_results(csv_path, run_name=f"benchmark/{model_name}"))
    return pd.concat(dfs, ignore_index=True)


def prepare_leaderboard(
    local_df: pd.DataFrame,
    benchmark_df: pd.DataFrame,
    metrics: list[str],
    datasets: list[str] | None = None,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    combined = pd.concat([local_df, benchmark_df], ignore_index=True)
    if datasets is None:
        local_datasets = set(local_df["dataset"].unique().tolist())
        benchmark_datasets = set(benchmark_df["dataset"].unique().tolist())
        datasets = sorted(
            local_datasets.intersection(benchmark_datasets)
            if not benchmark_df.empty
            else local_datasets
        )
    combined = combined[combined["dataset"].isin(datasets)].copy()
    if combined.empty:
        raise ValueError("No matching dataset rows found for the selected scope.")

    leaderboard = (
        combined.groupby(["run_name", "model"], as_index=False)[metrics]
        .mean()
        .sort_values(metrics)
        .reset_index(drop=True)
    )
    dataset_counts = (
        combined.groupby(["run_name", "model"], as_index=False)["dataset"]
        .nunique()
        .rename(columns={"dataset": "n_datasets"})
    )
    leaderboard = leaderboard.merge(dataset_counts, on=["run_name", "model"], how="left")

    for metric in metrics:
        leaderboard[f"{metric}_rank"] = leaderboard[metric].rank(method="dense")

    per_dataset = (
        combined[["dataset", "run_name", "model", *metrics]]
        .sort_values(["dataset", *metrics])
        .reset_index(drop=True)
    )
    return leaderboard, per_dataset


In [ ]:
local_df = discover_local_results(LOCAL_RESULTS_ROOT, run_names=LOCAL_RUN_NAMES)
benchmark_df = load_benchmark_results(BENCHMARK_RESULTS_ROOT, BENCHMARK_MODELS)

leaderboard_df, per_dataset_df = prepare_leaderboard(
    local_df=local_df,
    benchmark_df=benchmark_df,
    metrics=METRICS,
    datasets=DATASETS,
)

leaderboard_df


In [ ]:
rank_cols = [f"{metric}_rank" for metric in METRICS]
overall_df = leaderboard_df.copy()
overall_df["avg_rank"] = overall_df[rank_cols].mean(axis=1)
overall_df = overall_df.sort_values(["avg_rank", *METRICS]).reset_index(drop=True)
overall_df[["run_name", "model", "avg_rank", *METRICS, *rank_cols]]


In [ ]:
per_dataset_df


In [ ]:
# Optional: save the generated tables for later use.
output_dir = Path("experiments/gift-eval/results/leaderboards")
output_dir.mkdir(parents=True, exist_ok=True)
leaderboard_df.to_csv(output_dir / "leaderboard_summary.csv", index=False)
per_dataset_df.to_csv(output_dir / "leaderboard_per_dataset.csv", index=False)
output_dir
